In [1]:
# Parameters
run_date = ""  # "" -> сьогодні; papermill: -p run_date 2026-07-28
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# --- параметри стратегії ---
STRATEGY_CODE  = "sector_corr"
lookback_years = 3      # ковзне вікно історії від run_date
min_days       = 5      # мін. спільних звітних днів, щоб рахувати кореляцію
value_col      = "gap_div"

# Поріг для ФАЙЛУ, не для розрахунку. Рахуємо всі пари (summary.csv рахує
# mean/median по всьому спектру і без слабких пар був би зміщений), але в
# sector_corr.csv.gz пишемо лише |corr| >= min_abs_corr: споживач (CORR-фільтр
# у бриджі) нижче цього порога не питає ніколи, а решта — 85% рядків, які
# їдуть по мережі і лежать у пам'яті без жодного застосування.
min_abs_corr   = 0.5

# джерело даних: "datum" (як CRACEN/ArbitRage) або "sql" (py_common + database.ini)
DATA_SOURCE = os.environ.get("ORION_SECTOR_CORR_SOURCE", "datum")
DB_INI      = os.environ.get("ORION_DB_INI", "database.ini")

# ensure output exists
os.makedirs(output_dir, exist_ok=True)

In [2]:
# Import basic modules
import os
import json
import datetime
from datetime import timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from datum_api_client import DatumApi

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [3]:
# --- ініціалізація DatumApi ------------------------------------------
# DatumApi тримає імена своїх json-ів відносними і читає їх від cwd:
#     config_file = 'datum_api_config.json'
# Якщо файлу в cwd немає, read_from_file() повертає None, і init() падає на
#     TypeError: 'NoneType' object is not subscriptable
# У пайплайні це не видно (papermill стартує з cwd=ORION_HOME, куди
# run_orion_daily.py стейджить секрети), але при ручному запуску з notebooks/
# ламається. Тому вказуємо шляхи явно.
CFG_NAME, CRED_NAME, TOKEN_NAME = (
    "datum_api_config.json", "datum_api_credentials.json", "access_token.json")


def _datum_search_dirs():
    dirs = []
    for env_name in ("DATUM_API_CONFIG_PATH", "DATUM_CONFIG_PATH", "DATUM_API_CFG_PATH"):
        v = os.environ.get(env_name)
        if v:
            p = Path(v).expanduser()
            dirs.append(p.parent if p.suffix == ".json" else p)
    if config_path:
        p = Path(config_path).expanduser()
        dirs.append(p.parent if p.suffix == ".json" else p)
    for env_name in ("DATUM_HOME", "ORION_HOME"):
        v = os.environ.get(env_name)
        if v:
            dirs.append(Path(v).expanduser())
    here = Path.cwd().resolve()
    dirs.append(here)
    dirs.extend(here.parents)

    seen, out = set(), []
    for d in dirs:
        try:
            d = d.expanduser().resolve()
        except Exception:
            continue
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def _resolve_datum_dir() -> Path:
    checked = _datum_search_dirs()
    for d in checked:                       # 1) повний комплект
        if (d / CFG_NAME).exists() and (d / CRED_NAME).exists():
            return d
    for d in checked:                       # 2) хоча б конфіг (токен ще живий)
        if (d / CFG_NAME).exists():
            print(f"warning: {CRED_NAME} не знайдено поруч із конфігом у {d}")
            return d
    raise FileNotFoundError(
        f"Не знайдено {CFG_NAME}. Перевірені каталоги:\n"
        + "\n".join(f"  - {d}" for d in checked[:15])
        + "\nВкажи DATUM_API_CONFIG_PATH (шлях до json) або DATUM_HOME (каталог)."
    )


DATUM_DIR = _resolve_datum_dir()
DatumApi.config_file      = str(DATUM_DIR / CFG_NAME)
DatumApi.credentials_file = str(DATUM_DIR / CRED_NAME)
DatumApi.token_file       = str(DATUM_DIR / TOKEN_NAME)   # DatumApi пише сюди оновлений токен
DatumApi.is_init = False
DatumApi.init()

print("Datum secrets:", DATUM_DIR)
print("  api_domain :", DatumApi.api_domain)

Datum secrets: C:\datum-api-examples-main
  api_domain : https://api.datum-rd.com


In [4]:
def _resolve_signals_dir(strategy_code: str) -> Path:
    """Каталог для вихідних файлів: <signals>/<strategy_code>.

    Пріоритет: SIGNALS_DIR -> ORION_HOME/signals -> output_dir (комірка Parameters)
    -> пошук папки OriON вгору по дереву. Той самий контракт, що у ArbitRage,
    але без вимоги CRACEN/final.parquet — ця стратегія його не споживає.
    """
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    signals_base = None
    if sig_env:
        signals_base = Path(sig_env).expanduser().resolve()
    elif orion_env:
        signals_base = (Path(orion_env).expanduser().resolve() / "signals").resolve()
    elif output_dir:
        signals_base = Path(output_dir).expanduser().resolve()

    if signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME or SIGNALS_DIR.")
        signals_base = (orion_home / "signals").resolve()

    out = (signals_base / strategy_code.lower()).resolve()
    out.mkdir(parents=True, exist_ok=True)
    return out


OUT_DIR = _resolve_signals_dir(STRATEGY_CODE)
print("Output dir:", OUT_DIR)

Output dir: C:\datum-api-examples-main\OriON\signals\sector_corr


In [5]:
# --- вікно історії ---------------------------------------------------
def _resolve_run_date(value) -> datetime.date:
    v = str(value or "").strip() or os.environ.get("ORION_RUN_DATE", "").strip()
    if not v:
        return datetime.date.today()
    return pd.to_datetime(v).date()


end_date = _resolve_run_date(run_date)

try:
    from dateutil.relativedelta import relativedelta
    start_date = end_date - relativedelta(years=int(lookback_years))
except Exception:
    start_date = end_date.replace(year=end_date.year - int(lookback_years))

start_date_str = f"{start_date:%Y-%m-%d}"
end_date_str   = f"{end_date:%Y-%m-%d}"

print(f"Window: {start_date_str} -> {end_date_str}  ({lookback_years}y, source={DATA_SOURCE})")

Window: 2023-09-03 -> 2026-09-03  (3y, source=datum)


In [6]:
# --- мапа сектор (lvl2) -> бенчмарк-ETF ------------------------------
NO_ETF = "no etf"

BENCHMARK_GROUPS = {
    "SPY": ["Health Care", "Industrial Products", "Real Estate", "Media", "Telecommunications"],
    "QQQ": ["Tech Hardware & Semiconductors", "Software & Tech Services"],
    "IWM": ["Industrial Services", "Retail & Whsle - Discretionary", "Consumer Staple Products",
            "Consumer Discretionary Products", "Consumer Discretionary Services",
            "Retail & Wholesale - Staples"],
    "XLF": ["Financial Services", "Banking", "Insurance", "Specialty Finance"],
    "XLU": ["Utilities"],
    "XLE": ["Oil & Gas"],
    NO_ETF: ["Materials", "Renewable Energy"],
}
ETFS = ["SPY", "QQQ", "IWM", "XLF", "XLU", "XLE"]

# Будь-яка група, що не є реальним ETF, зводиться до сентинела NO_ETF.
# В оригіналі група звалась "No_ETF" і не збігалася з "no etf" -> Materials
# та Renewable Energy йшли гілкою "мінус ETF", не знаходили бенчмарк
# і отримували gap_div = NaN, тобто випадали з кореляцій повністю.
benchmark_map = {lvl2: (etf if etf in ETFS else NO_ETF)
                 for etf, lst in BENCHMARK_GROUPS.items() for lvl2 in lst}
print("lvl2 mapped:", len(benchmark_map),
      "| без ETF:", sorted(k for k, v in benchmark_map.items() if v == NO_ETF))

lvl2 mapped: 21 | без ETF: ['Materials', 'Renewable Energy']


In [7]:
# --- шар доступу до даних --------------------------------------------
# Дві реалізації з однаковим контрактом:
#   fetch_reports(tickers) -> DataFrame[ticker, date]        (дата реакції на звіт)
#   fetch_gaps(tickers)    -> DataFrame[ticker, date, gap]
# "datum" — як у CRACEN/ArbitRage, працює в пайплайні run_orion_daily.py.
# "sql"   — оригінальний прямий доступ до БД (потрібні py_common + database.ini).

_conn = None


def _get_conn():
    """Ліниве підключення до БД лише для DATA_SOURCE='sql'."""
    global _conn
    if _conn is not None:
        return _conn
    import py_common.repository as repository
    import py_common.holidays as holi
    ini = Path(DB_INI)
    if not ini.is_absolute() and not ini.exists():
        for base in [Path.cwd(), Path(os.environ.get("ORION_HOME", ".")), Path.cwd() / "ops"]:
            cand = (base / DB_INI).resolve()
            if cand.exists():
                ini = cand
                break
    if not Path(ini).exists():
        raise FileNotFoundError(f"database.ini not found: {DB_INI}. Set ORION_DB_INI.")
    _conn = repository.create_conn(str(ini))
    holi.Holidays.init_holidays(_conn)
    return _conn


def _norm_date(s):
    return pd.to_datetime(s, errors="coerce").dt.strftime("%Y-%m-%d")


def _pick_col(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None


# ---------- DATUM ----------
def _datum_reports_one(ticker):
    try:
        df = DatumApi.data_request("/reports", {
            "ticker": str(ticker),
            "start_move_date": start_date_str,
            "end_move_date": end_date_str,
        })
        if df is None or df.empty:
            return None
        col = _pick_col(df, ["move_date", "reaction_date", "report_date", "date", "dt"])
        if col is None:
            return None
        out = pd.DataFrame({"ticker": str(ticker).upper(), "date": _norm_date(df[col])})
        return out.dropna(subset=["date"]).drop_duplicates()
    except Exception as e:
        print(f"reports failed {ticker}: {e}")
        return None


def _datum_gaps_one(ticker):
    try:
        df = DatumApi.data_request("/daily/gaps", {
            "ticker": str(ticker),
            "start_date": start_date_str,
            "end_date": end_date_str,
            "format": "json_records",
        })
        if df is None or df.empty:
            return None
        dcol = _pick_col(df, ["date", "move_date", "dt", "datetime", "day"])
        if dcol is None or "gap" not in df.columns:
            return None
        out = pd.DataFrame({
            "ticker": str(ticker).upper(),
            "date": _norm_date(df[dcol]),
            "gap": pd.to_numeric(df["gap"], errors="coerce"),
        })
        return out.dropna(subset=["date", "gap"])
    except Exception as e:
        print(f"gaps failed {ticker}: {e}")
        return None


def _datum_many(tickers, fn, desc, max_workers=16):
    parts = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(fn, t) for t in tickers]
        done = 0
        for f in as_completed(futures):
            r = f.result()
            if r is not None and not r.empty:
                parts.append(r)
            done += 1
            if done % 1000 == 0:
                print(f"  {desc}: {done}/{len(futures)}", flush=True)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


# ---------- SQL ----------
def _sql_reports(tickers):
    q = """
    SELECT ticker_by_esignal AS ticker,
           announcement_date,
           holidays.reaction_date(announcement_date, announcement_time) AS reaction_date
    FROM tickers_by_company t
    JOIN earnings_date_history e ON t.id_company = e.id_company
    JOIN ticker_by_sorter tbs ON tbs.id_ticker = t.id
    JOIN sorter s ON s.id = tbs.id_sorter
    WHERE announcement_date >= %(start_date)s AND announcement_date <= %(end_date)s
    """
    df = pd.read_sql(q, _get_conn(), params={"start_date": start_date_str, "end_date": end_date_str})
    out = pd.DataFrame({"ticker": df["ticker"].astype(str).str.upper(),
                        "date": _norm_date(df["reaction_date"])})
    return out.dropna(subset=["date"]).drop_duplicates()


def _sql_gaps(tickers):
    q = """
    SELECT ticker_by_esignal AS ticker, date, open, prev_close
    FROM day d
    JOIN tickers_by_company tbc ON tbc.id = d.id_ticker
    WHERE tbc.ticker_by_esignal IN %(tickers)s
      AND date >= %(start_date)s AND date <= %(end_date)s
    """
    df = pd.read_sql(q, _get_conn(), params={"tickers": tuple(tickers),
                                             "start_date": start_date_str,
                                             "end_date": end_date_str})
    df["gap"] = ((df["open"] / df["prev_close"] - 1) * 100).round(2)
    out = pd.DataFrame({"ticker": df["ticker"].astype(str).str.upper(),
                        "date": _norm_date(df["date"]),
                        "gap": pd.to_numeric(df["gap"], errors="coerce")})
    return out.dropna(subset=["date", "gap"])


def fetch_reports(tickers):
    if DATA_SOURCE == "sql":
        return _sql_reports(tickers)
    return _datum_many(tickers, _datum_reports_one, "reports")


def fetch_gaps(tickers):
    if DATA_SOURCE == "sql":
        return _sql_gaps(tickers)
    return _datum_many(tickers, _datum_gaps_one, "gaps")

In [8]:
# --- всесвіт тікерів --------------------------------------------------
tickers_df = DatumApi.data_request("/tickers", {
    "fields": "market_cap,shares_float,lvl2",
    "active": True,
    "us_exchange": True,
    "listed": True,
})
tickers_df = tickers_df.dropna()
tickers_df["ticker"] = tickers_df["ticker"].astype(str).str.upper()
tickers_df = tickers_df.drop_duplicates("ticker")

UNIVERSE = tickers_df["ticker"].tolist()
print("Tickers:", len(UNIVERSE))

Tickers: 5415


In [9]:
# --- звіти ------------------------------------------------------------
reports_df = fetch_reports(UNIVERSE)
if reports_df.empty:
    raise RuntimeError("No reports fetched — перевір джерело даних / вікно дат.")
reports_df["report?"] = "yes"
reports_df = reports_df.drop_duplicates(["ticker", "date"])
print("Report rows:", len(reports_df), "| tickers with reports:", reports_df["ticker"].nunique())

  reports: 1000/5415


  reports: 2000/5415


reports failed JKS: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JKS&start_move_date=2023-09-03&end_move_date=2026-09-03reports failed JL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JLHL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JLHL&start_move_date=2023-09-03&end_move_date=2026-09-03

reports failed JLL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JLL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JMKE: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JMKE&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JMIA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JMIA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JOBY: 429 Clien

reports failed JXG: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JXG&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JXN: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JXN&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JWEL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JWEL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KAI: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KAI&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed JYD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=JYD&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KARD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KARD&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KALU: 429 Clien

reports failed KDP: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KDP&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KD&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KC: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KC&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KDK: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KDK&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KELYA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KELYA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KEP: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=KEP&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed KEN: 429 Client Err

...

reports failed XSLL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XSLL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XRX: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XRX&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XTERU: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XTERU&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XTNT: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XTNT&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XYL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XYL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XTIA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=XTIA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed XWEL: 429

reports failed YDKG: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YDKG&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YHGJ: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YHGJ&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YIBO: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YIBO&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YI: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YI&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YEXT: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YEXT&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YHNA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YHNA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YETI: 429 C

reports failed ZBH: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZBH&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed YTRA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=YTRA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZBIO: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZBIO&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZCMD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZCMD&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZBRA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZBRA&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZD&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZDGE: 429 Cli

reports failed ZLAB: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZLAB&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZONE: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZONE&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZM: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZM&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZOOZ: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZOOZ&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZNTL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZNTL&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZTO: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/reports?ticker=ZTO&start_move_date=2023-09-03&end_move_date=2026-09-03
reports failed ZS: 429 Clien

Report rows: 23642 | tickers with reports: 2213


In [10]:
# --- гепи + бенчмарк + gap_div ---------------------------------------
gaps_df = fetch_gaps(UNIVERSE + ETFS)
if gaps_df.empty:
    raise RuntimeError("No gaps fetched — перевір джерело даних / вікно дат.")
gaps_df = gaps_df.drop_duplicates(["ticker", "date"])
print("Gap rows:", len(gaps_df))

gaps_df = gaps_df.merge(tickers_df[["ticker", "lvl2"]], how="left", on="ticker")
gaps_df["benchmark"] = gaps_df["lvl2"].map(benchmark_map).fillna(NO_ETF)

etf_gaps_df = (gaps_df[gaps_df["ticker"].isin(ETFS)][["ticker", "date", "gap"]]
               .rename(columns={"ticker": "benchmark", "gap": "main_etf_gap"}))
gaps_df = gaps_df.merge(etf_gaps_df, how="left", on=["date", "benchmark"])

# Якщо бенчмарк є, але його гепу на цю дату немає (свято/халт по ETF) —
# відкидаємо рядок, а не мовчки лишаємо NaN у gap_div.
has_etf = gaps_df["benchmark"] != NO_ETF
missing_etf = int((has_etf & gaps_df["main_etf_gap"].isna()).sum())
if missing_etf:
    print(f"  dropped {missing_etf} rows: benchmark gap missing for that date")
    gaps_df = gaps_df[~(has_etf & gaps_df["main_etf_gap"].isna())]
    has_etf = gaps_df["benchmark"] != NO_ETF

gaps_df["gap_div"] = np.where(has_etf, gaps_df["gap"] - gaps_df["main_etf_gap"], gaps_df["gap"])

gaps_df = gaps_df.merge(reports_df[["ticker", "date", "report?"]], how="left", on=["ticker", "date"])
gaps_df["report?"] = gaps_df["report?"].fillna("no")

gaps_df = gaps_df.dropna(subset=["lvl2"])
gaps_df = gaps_df.sort_values(["lvl2", "ticker", "date"]).reset_index(drop=True)
print("Rows:", len(gaps_df), "| lvl2:", gaps_df["lvl2"].nunique(),
      "| report=yes:", int((gaps_df["report?"] == "yes").sum()))

gaps failed AARD: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AARD&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed AAP: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AAP&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed AAME: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AAME&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed AAMI: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AAMI&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed AADX: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AADX&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed AA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=AA&start_date=2023-09-03&end_date

gaps failed ABCB: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABCB&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ABEO: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABEO&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ABCL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABCL&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ABEV: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABEV&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ABBV: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABBV&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ABTS: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ABTS&start_date=2023-09-03&en

gaps failed ACGL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACGL&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACH: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACH&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACHC: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACHC&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACHV: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACHV&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACHR: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACHR&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACI: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACI&start_date=2023-09-03&end_da

gaps failed ACTG: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACTG&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACRV: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACRV&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACTU: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACTU&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACU: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACU&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ACVA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ACVA&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADBE: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADBE&start_date=2023-09-03&end_

gaps failed ADPT: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADPT&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADP: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADP&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADT: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADT&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADMA: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADMA&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADNT: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADNT&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed ADUR: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=ADUR&start_date=2023-09-03&end_da

...

gaps failed MVIS: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MVIS&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed MVST: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MVST&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed MWYN: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MWYN&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed MWH: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MWH&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed MWG: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MWG&start_date=2023-09-03&end_date=2026-09-03&format=json_records
gaps failed MXL: 429 Client Error: Too Many Requests for url: https://api.datum-rd.com/daily/gaps?ticker=MXL&start_date=2023-09-03&end_date

  gaps: 4000/5421


  gaps: 5000/5421


Gap rows: 1453482


Rows: 1448970 | lvl2: 20 | report=yes: 0


In [11]:
# --- кореляція тікера з однонсекторними peer-ами на його звітних днях --
# Векторизовано через pivot по кожному lvl2: результат ідентичний
# поцикловій версії, але без O(n^2) фільтрації по всьому DataFrame.
def run_sector_correlations(df, value_col="gap_div", min_days=5):
    need = ["ticker", "date", "lvl2", "report?", value_col]
    d = df[need].copy()

    first_lvl2 = d.groupby("ticker")["lvl2"].first()
    rep = d[d["report?"] == "yes"]
    rep_dates = rep.groupby("ticker")["date"].apply(lambda s: pd.Index(s.unique()))
    n_reports = rep.groupby("ticker")["date"].nunique()

    def _stub(t, lvl2, n, status):
        return pd.DataFrame([{"ticker": t, "peer_ticker": None, "lvl2": lvl2,
                              "n_days": int(n), "correlation": np.nan, "corr_status": status}])

    frames = []
    for lvl2, g in d.groupby("lvl2", sort=False):
        piv = g.pivot_table(index="date", columns="ticker", values=value_col, aggfunc="first")
        targets = [t for t in g["ticker"].unique() if n_reports.get(t, 0) >= min_days]
        for t in targets:
            if t not in piv.columns:
                continue
            sub = piv.loc[piv.index.intersection(rep_dates[t])]
            tgt = sub[t]
            peers = sub.drop(columns=[t])
            if peers.shape[1] == 0:
                frames.append(_stub(t, lvl2, n_reports[t], "not_enough_peer_overlap"))
                continue
            n_ok = peers.notna().mul(tgt.notna(), axis=0).sum()
            keep = n_ok[n_ok >= min_days].index
            if len(keep) == 0:
                frames.append(_stub(t, lvl2, n_reports[t], "not_enough_peer_overlap"))
                continue
            corr = peers[keep].corrwith(tgt)
            frames.append(pd.DataFrame({"ticker": t, "peer_ticker": keep, "lvl2": lvl2,
                                        "n_days": n_ok[keep].to_numpy(),
                                        "correlation": corr.to_numpy(), "corr_status": "ok"}))

    short = [t for t in d["ticker"].unique() if n_reports.get(t, 0) < min_days]
    if short:
        frames.append(pd.DataFrame({
            "ticker": short, "peer_ticker": None,
            "lvl2": [first_lvl2.get(t) if n_reports.get(t, 0) > 0 else None for t in short],
            "n_days": [int(n_reports.get(t, 0)) for t in short],
            "correlation": np.nan, "corr_status": "not_enough_reports"}))

    return pd.concat(frames, ignore_index=True)

In [12]:
# --- розрахунок -------------------------------------------------------
sector_corr_df = run_sector_correlations(gaps_df, value_col=value_col, min_days=int(min_days))
sector_corr_df = sector_corr_df.sort_values(["ticker", "correlation"],
                                            ascending=[True, False]).reset_index(drop=True)

print("Pairs:", len(sector_corr_df))
print(sector_corr_df["corr_status"].value_counts().to_string())

Pairs: 2154
not_enough_reports    2154


In [13]:
# --- зведення по тікеру ----------------------------------------------
ok = sector_corr_df[sector_corr_df["corr_status"] == "ok"]

if ok.empty:
    summary_df = pd.DataFrame(columns=["ticker", "lvl2", "n_peers", "mean_corr", "median_corr",
                                       "best_peer", "best_corr", "worst_peer", "worst_corr"])
else:
    idx_best  = ok.groupby("ticker")["correlation"].idxmax()
    idx_worst = ok.groupby("ticker")["correlation"].idxmin()
    agg = ok.groupby("ticker").agg(lvl2=("lvl2", "first"),
                                   n_peers=("peer_ticker", "nunique"),
                                   mean_corr=("correlation", "mean"),
                                   median_corr=("correlation", "median"))
    summary_df = (agg
                  .join(ok.loc[idx_best].set_index("ticker")[["peer_ticker", "correlation"]]
                        .rename(columns={"peer_ticker": "best_peer", "correlation": "best_corr"}))
                  .join(ok.loc[idx_worst].set_index("ticker")[["peer_ticker", "correlation"]]
                        .rename(columns={"peer_ticker": "worst_peer", "correlation": "worst_corr"}))
                  .reset_index())

print("Summary rows:", len(summary_df))
summary_df.head()

Summary rows: 0


,ticker,lvl2,n_peers,mean_corr,median_corr,best_peer,best_corr,worst_peer,worst_corr


In [14]:
# --- запис у signals/<strategy>/ -------------------------------------
# Пуш на GitHub робить run_orion_daily.py (крок 6): він копіює весь signals/
# у репозиторій OriON-stats і комітить. Тут лише пишемо файли.
#
# ФОРМАТ sector_corr.csv.gz — заточений під один запит, який його читає:
# "S сьогодні звітує — кого відсікти разом із ним".
#
#   ticker,peer_ticker,correlation
#
#   * НАПРЯМОК ЗНАЧУЩИЙ. Кореляція рахується на звітних днях `ticker` (див.
#     run_sector_correlations: piv.loc[rep_dates[t]]), тому corr(A->B) і
#     corr(B->A) — різні величини на різних наборах дат, а не дзеркало. Читати
#     треба саме рядки з ticker == тікер, що звітує сьогодні.
#   * Три колонки. lvl2 виводиться з тікера, n_days/corr_status — діагностика
#     розрахунку; споживачу потрібні лише пари й сила зв'язку.
#   * Знак збережено: |corr| вирішує відсічення, але знак каже, в який бік
#     поїде peer, і колись знадобиться. Округлення до 4 знаків — далі йде
#     шум оцінки на 5-21 спостереженні.
#   * Рядки згруповані по ticker і всередині відсортовані за спаданням |corr|,
#     тож скан може зупинитись на першому значенні нижче свого порога.
#
# Повний спектр лишається в summary.csv (mean/median/best/worst рахуються до
# відсічення) і в meta.json — так видно, скільки саме відкинуто.
pairs_out = sector_corr_df[
    (sector_corr_df["corr_status"] == "ok")
    & sector_corr_df["correlation"].abs().ge(float(min_abs_corr))
].copy()

pairs_out["correlation"] = pairs_out["correlation"].round(4)
pairs_out = (pairs_out
             .assign(_abs=pairs_out["correlation"].abs())
             .sort_values(["ticker", "_abs"], ascending=[True, False])
             .drop(columns=["_abs"])[["ticker", "peer_ticker", "correlation"]]
             .reset_index(drop=True))

print(f"Pairs written: {len(pairs_out):,} / {len(sector_corr_df):,} "
      f"(|corr| >= {min_abs_corr}) | tickers: {pairs_out['ticker'].nunique():,}")

meta = {
    "strategy": STRATEGY_CODE,
    "run_date": end_date_str,
    "start_date": start_date_str,
    "end_date": end_date_str,
    "lookback_years": int(lookback_years),
    "min_days": int(min_days),
    "value_col": value_col,
    "data_source": DATA_SOURCE,
    "tickers_universe": int(len(UNIVERSE)),
    "gap_rows": int(len(gaps_df)),
    "report_rows": int(len(reports_df)),
    "pairs": int(len(sector_corr_df)),
    "pairs_ok": int((sector_corr_df["corr_status"] == "ok").sum()),
    "tickers_with_corr": int(sector_corr_df.loc[sector_corr_df["corr_status"] == "ok", "ticker"].nunique()),
    # Що саме лежить у sector_corr.csv.gz після відсічення — щоб споживач міг
    # перевірити поріг, а не здогадуватись про нього за даними.
    "min_abs_corr": float(min_abs_corr),
    "pairs_written": int(len(pairs_out)),
    "tickers_written": int(pairs_out["ticker"].nunique()),
    "pairs_columns": ["ticker", "peer_ticker", "correlation"],
    "pairs_direction": "correlation measured on the report days of `ticker`; not symmetric",
    "generated_at_utc": datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
}

if dry_run:
    print("dry_run=True — файли не записані")
    print(json.dumps(meta, indent=2, ensure_ascii=False))
else:
    p_pairs   = OUT_DIR / "sector_corr.csv.gz"
    p_summary = OUT_DIR / "summary.csv"
    p_meta    = OUT_DIR / "meta.json"

    pairs_out.to_csv(p_pairs, index=False, compression="gzip")
    summary_df.to_csv(p_summary, index=False)
    p_meta.write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    for p in (p_pairs, p_summary, p_meta):
        print(f"  wrote {p}  ({p.stat().st_size:,} bytes)")

print("SectorCorr completed.")

Pairs written: 0 / 2,154 (|corr| >= 0.5) | tickers: 0
  wrote C:\datum-api-examples-main\OriON\signals\sector_corr\sector_corr.csv.gz  (70 bytes)
  wrote C:\datum-api-examples-main\OriON\signals\sector_corr\summary.csv  (85 bytes)
  wrote C:\datum-api-examples-main\OriON\signals\sector_corr\meta.json  (654 bytes)
SectorCorr completed.
